#  **Data Collection and Preprocessing**

### Import Libraries

In [53]:
import sys
import os

# Add parent directory to path so we can import from src
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

In [54]:
from src.preprocessing import (
    clean_review_text,
    display_app_info,
    review_dataframe,
    remove_duplicates,
    handle_missing_data,
    normalize_dates,
    validate_rating,
    preprocessing_report,
    save_cleaned_data,
)
from src.data_scrapping import scrap_reviews

### Web Scraping

#### App metadata

In [55]:
CBE_APP_ID = 'com.combanketh.mobilebanking'
display_app_info(CBE_APP_ID)

Commercial Bank of Ethiopia App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.2877455
Total Ratings: 48,323
Total Reviews: 9,307
Installs     : 5,000,000+


#### Scrape reviews

In [56]:
reviews = scrap_reviews(app_id=CBE_APP_ID)

Scraping reviews for com.combanketh.mobilebanking...
Collected 700 raw reviews


#### Collect review text, rating, review date, bank , source

In [57]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(reviews[0].keys()))

print("\nFirst raw review (sample):")
for key, value in reviews[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 0e691771-0024-4326-b71f-1685b91dc83d
  userName: Natua 123
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocKB__m7jjhJeriNmBYWHKGDUkX8SML-vVdHjz97iLakm9Lbig=mo
  content: incredible
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 5.3.0
  at: 2026-05-14 10:53:56
  replyContent: None
  repliedAt: None
  appVersion: 5.3.0


In [58]:
df = review_dataframe(reviews, app_info={'title': 'CBE Bank'})

print(f"Shape: {df.shape}")
df.head()

Shape: (700, 6)


,review_id,review,rating,date,bank,source
0,0e691771-0024-4326-b71f-1685b91dc83d,incredible,5,2026-05-14 10:53:56,CBE Bank,Google Play
1,982b1262-3d54-4a12-811a-8b26b7ecc777,best app for financial sector,5,2026-05-14 05:44:01,CBE Bank,Google Play
2,06f6640c-b65c-43e4-88ef-0a79be8b9534,it's a good application,5,2026-05-13 20:28:58,CBE Bank,Google Play
3,eda74236-f7f3-4422-a139-a181e832bc27,thank you cbe,5,2026-05-13 17:16:37,CBE Bank,Google Play
4,ff53332b-2e76-46d6-83d3-f93f968a4b18,is good,5,2026-05-13 16:18:45,CBE Bank,Google Play


## Preprocessing

#### Remove duplicate reviews

In [59]:
df_clean = df.copy()

In [60]:
df_clean = remove_duplicates(df_clean)

Removed 0 duplicate reviews
Remaining: 700 reviews


#### Handle missing values

In [61]:
df_clean =handle_missing_data(df_clean)

Removed 0 rows with missing critical data
Remaining: 700 reviews


#### Normalize dates to YYYY-MM-DD format

In [62]:
df_clean = normalize_dates(df_clean)

Before normalization:
0   2026-05-14 10:53:56
1   2026-05-14 05:44:01
2   2026-05-13 20:28:58
dtype: datetime64[us]

After normalization:
0    2026-05-14
1    2026-05-14
2    2026-05-13
dtype: str

Date range: 2026-02-03 to 2026-05-14


#### Handle incorrect ratings

In [63]:
df_clean = validate_rating(df_clean)

All ratings are valid (1-5).
Remaining: 700 reviews


#### Clean review text

In [64]:
df['review'] = df['review'].apply(clean_review_text)

print("Sample cleaned reviews:")
print(df['review'].head(10).to_string())

Sample cleaned reviews:
0                                           incredible
1                        best app for financial sector
2                              it's a good application
3                                        thank you cbe
4                                              is good
5                                                  wow
6                                     good application
7    nice, but i can't get some recently transactio...
8    very secure but very poor interface and limite...
9                                       very nice 100%


#### Save the cleaned dataset

In [65]:
# Select only the 5 required columns in the right order
df_clean = df_clean[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (700, 5)


,review,rating,date,bank,source
0,incredible,5,2026-05-14,CBE Bank,Google Play
1,best app for financial sector,5,2026-05-14,CBE Bank,Google Play
2,it's a good application,5,2026-05-13,CBE Bank,Google Play
3,is good,5,2026-05-13,CBE Bank,Google Play
4,wow,5,2026-05-13,CBE Bank,Google Play
5,Good application,2,2026-05-13,CBE Bank,Google Play
6,"Nice, but I can't get some recently transactio...",5,2026-05-13,CBE Bank,Google Play
7,Very Secure but very poor interface and limite...,1,2026-05-13,CBE Bank,Google Play
8,very nice 100%,5,2026-05-13,CBE Bank,Google Play
9,thank you cbe,5,2026-05-13,CBE Bank,Google Play


In [66]:
save_cleaned_data(df_clean, output_path="../data/processed/cbe_reviews_cleaned.csv")

Cleaned data saved to ../data/processed/cbe_reviews_cleaned.csv
Saved to: ../data/processed/cbe_reviews_cleaned.csv


### Report

In [67]:
preprocessing_report(df, df_clean)

  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :    700
  Reviews after cleaning :    700
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2026-02-03  to  2026-05-14

  Rating distribution:
    5 stars :  472 (67.4%)  ██████████████████████████████████████████████████████████████████████████████████████████████
    4 stars :   59 ( 8.4%)  ███████████
    3 stars :   45 ( 6.4%)  █████████
    2 stars :   21 ( 3.0%)  ████
    1 stars :  103 (14.7%)  ████████████████████

  Text length stats:
    Min    : 1 characters
    Median : 14 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

